# 7장 머신러닝 API : 1차시 (Iris)
Iris 데이터로 모델을 학습하고, 파일로 저장한 뒤, Flask API 로 내보냅니다. 노트북은 `data_analysis` 폴더에서 엽니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import platform

fonts = {"Windows": "Malgun Gothic", "Darwin": "AppleGothic"}
plt.rcParams["font.family"] = fonts.get(platform.system(), "NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

## 1. 데이터 : Iris 150 송이

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
df.shape

In [ ]:
df.head()

In [ ]:
df["species"].value_counts()

In [ ]:
markers = {"setosa": "o", "versicolor": "s", "virginica": "^"}
for name, g in df.groupby("species"):
    plt.scatter(g["petal length (cm)"], g["petal width (cm)"],
                marker=markers[name], label=name, s=28)
plt.xlabel("petal length (cm)")
plt.ylabel("petal width (cm)")
plt.title("Iris : petal length vs width, by species")
plt.legend()
plt.show()

## 2. training set 과 test set 으로 나눈다

In [ ]:
from sklearn.model_selection import train_test_split

X = df[iris.feature_names]        # feature : 측정값 네 열
y = df["species"]                 # label : 품종
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
print("training :", len(X_train), " test :", len(X_test))

## 3. 학습 전에 : 사람이 그림을 보고 그은 규칙

In [ ]:
import numpy as np

def hand_rule(X):                     # 그림을 보고 사람이 정한 기준 두 개
    pl = X["petal length (cm)"]
    pw = X["petal width (cm)"]
    return np.where(pl <= 2.5, "setosa",
           np.where(pw <= 1.7, "versicolor", "virginica"))

print("training accuracy :", round((hand_rule(X_train) == y_train).mean(), 3))
print("test accuracy     :", round((hand_rule(X_test) == y_test).mean(), 3))

## 3. 학습 : decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=2, random_state=0)
model.fit(X_train, y_train)

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(9, 4.2))
plot_tree(model, feature_names=iris.feature_names,
          class_names=iris.target_names,
          impurity=False, fontsize=9)
plt.show()

## 4. 예측 : 새 꽃 한 송이

In [ ]:
new = pd.DataFrame([[6.0, 2.9, 4.5, 1.5]], columns=iris.feature_names)
print(model.predict(new)[0])

In [ ]:
print(model.predict_proba(new).round(2))    # setosa, versicolor, virginica 순서

## 5. 평가 : test set 30 송이로

In [ ]:
print("training accuracy :", round(model.score(X_train, y_train), 3))
print("test accuracy     :", round(model.score(X_test, y_test), 3))

In [ ]:
from sklearn.metrics import confusion_matrix

pred = model.predict(X_test)
pd.DataFrame(confusion_matrix(y_test, pred),
             index=["true_" + n for n in iris.target_names],
             columns=["pred_" + n for n in iris.target_names])

In [ ]:
X_test[pred != y_test].assign(실제=y_test[pred != y_test], 예측=pred[pred != y_test])

## 6. 저장 : 모델을 파일로

In [ ]:
import joblib

joblib.dump(model, "iris_model.pkl")

In [ ]:
loaded = joblib.load("iris_model.pkl")
print(loaded.predict(new)[0])